In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/Obesity.csv')

# EDA

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df["BMI"] = (df["Weight"] / (df["Height"] ** 2)).round(2)

In [ ]:
#Avaliar como está a dispersão do BMI (Body Mass Index)

import matplotlib.pyplot as plt


df['BMI'].plot(kind='box', figsize=(8, 6))

plt.title('Boxplot - BMI')
plt.ylabel('Índice de Massa Corporal (BMI)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
#Verificar se existe relação da coluna Obesity com BMI para evitar vazamento de dados
df.groupby("Obesity")["BMI"].mean()

Vou verificado que o BMI não iria mudar muita coisa, apenas seria uma feature a mais que causaria vazamento de dados em nosso desenvolvimento, então é melhor retirar ela e deixar apenas a coluna Obesity.

In [ ]:
#Removendo a coluna BMI e Weight para evitar vazamento de dados
df = df.drop(columns=['BMI', 'Weight'])

In [ ]:
df.info()

In [ ]:
#Transformar as váriaveis categoricas em numericas e transformar em colunas em categoricas usando OneHotEncoder além de usar StandartScaler para as variáveis numericas
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    'Age', 'Height',
    'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE'
]

categorical_features = [
    'Gender', 'family_history', 'FAVC',
    'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS'
]


preprocessor = ColumnTransformer(
    transformers=[

        ('num', StandardScaler(), numeric_features),

        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            drop='first',
            sparse_output=False
        ), categorical_features)
    ],


    remainder='drop'
)


print(f"   → {len(numeric_features)} colunas numéricas serão escaladas")
print(f"   → {len(categorical_features)} colunas categóricas serão one-hot encoded")

In [ ]:
  X = df.drop(columns=['Obesity'])
y = df['Obesity']

X_processed = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)

print("Shape final:", X_processed_df.shape)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(X_processed_df.head())

In [ ]:
y.head()

In [ ]:
# Mapeamento manual das classes
class_mapping = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}


y_encoded = y.map(class_mapping)

print("Classes mapeadas:")
print(y_encoded.value_counts().sort_index())

inverse_mapping = {v: k for k, v in class_mapping.items()}
print("\nMapeamento inverso:")
for num, nome in inverse_mapping.items():
    print(f"{num} → {nome}")

#Modelo XGBoost 88%

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import time
import numpy as np


class_mapping = {
    'Insufficient_Weight': 0, 'Normal_Weight': 1, 'Overweight_Level_I': 2,
    'Overweight_Level_II': 3, 'Obesity_Type_I': 4, 'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}

y_encoded = y.map(class_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

param_dist = {
    'n_estimators': [400, 600],
    'max_depth': [4, 6],
    'learning_rate': [0.05],
    'subsample': [0.75, 0.85],
    'colsample_bytree': [0.7, 0.8],
    'min_child_weight': [1, 5]
}

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=7,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2
)



random_search.fit(X_train, y_train)

print("\n Melhores Parâmetros:")
print(random_search.best_params_)

print(f"\n Melhor Acurácia (média CV 5 folds): {random_search.best_score_:.4f}")

print("\n Acurácias por Fold - Melhor configuração:")
best_index = random_search.best_index_

fold_scores = [
    random_search.cv_results_[f'split{i}_test_score'][best_index]
    for i in range(5)
]

for i, score in enumerate(fold_scores, 1):
    print(f"   Fold {i}: {score:.4f}")

print(f"\n Resumo:")
print(f"   Média:          {np.mean(fold_scores):.4f}")
print(f"   Desvio padrão:  {np.std(fold_scores):.4f}")
print(f"   Melhor fold:    {max(fold_scores):.4f}")
print(f"   Pior fold:      {min(fold_scores):.4f}")



In [ ]:

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)


print(f" Acurácia no Teste (hold-out): {accuracy_score(y_test, y_pred):.4f}\n")

print(classification_report(y_test, y_pred,
                            target_names=list(class_mapping.keys()),
                            digits=4))

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

class_names = list(class_mapping.keys())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Matriz de Confusão - Melhor Modelo XGBoost\n(Conjunto de Teste)', fontsize=14, pad=20)
plt.xlabel('Classe Predita')
plt.ylabel('Classe Real')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, values_format='d', xticks_rotation=45)
plt.title('Matriz de Confusão - Melhor Modelo XGBoost\n(Conjunto de Teste)')
plt.tight_layout()
plt.show()

# Random Forest Classifier 84%


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb


param_dist = {
    'n_estimators': [400, 600],
    'max_depth': [4, 6],
    'learning_rate': [0.05],
    'subsample': [0.75, 0.85],
    'colsample_bytree': [0.7, 0.8],
    'min_child_weight': [1, 5]
}

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=7,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X_train, y_train)


best_model_xg = random_search.best_estimator_

print("\n XGBoost treinado com sucesso!")
print("Melhor acurácia CV:", random_search.best_score_)
print("Tipo do modelo:", type(best_model_xg).__name__)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


param_grid = {
    'n_estimators': [300, 700],
    'max_depth': [6, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search_rf.fit(X_train, y_train)

best_rf_model = grid_search_rf.best_estimator_


print("Melhor acurácia CV:", grid_search_rf.best_score_)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

y_pred = best_rf_model.predict(X_test)

inverse_mapping = {v: k for k, v in class_mapping.items()}
y_test_names = np.array([inverse_mapping[i] for i in y_test])
y_pred_names = np.array([inverse_mapping[i] for i in y_pred])

cm = confusion_matrix(y_test_names, y_pred_names, labels=list(class_mapping.keys()))

plt.figure(figsize=(11, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_mapping.keys(),
            yticklabels=class_mapping.keys())
plt.title('Matriz de Confusão - Random Forest')
plt.show()

# Regressão Logística 60%

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score
import time
import numpy as np

print("Iniciando RandomizedSearchCV para Regressão Logística (Versão Corrigida)...\n")

param_dist = {
    'C': [0.01, 0.1, 50],
    'penalty': ['l2', None],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [1000, 2000]
}

log_reg = LogisticRegression(
    multi_class='multinomial',
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1,
    error_score='raise'
)

random_search.fit(X_train, y_train)

print("\n Melhores Parâmetros:")
print(random_search.best_params_)

print(f"\n Melhor Acurácia (média CV 5 folds): {random_search.best_score_:.4f}")

print("\n Acurácias por Fold - Melhor configuração:")
best_index = random_search.best_index_

fold_scores = [
    random_search.cv_results_[f'split{i}_test_score'][best_index]
    for i in range(5)
]

for i, score in enumerate(fold_scores, 1):
    print(f"   Fold {i}: {score:.4f}")

print(f"\n Resumo:")
print(f"   Média:          {np.mean(fold_scores):.4f}")
print(f"   Desvio padrão:  {np.std(fold_scores):.4f}")
print(f"   Melhor fold:    {max(fold_scores):.4f}")
print(f"   Pior fold:      {min(fold_scores):.4f}")

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*60)
print(" AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (20%)")
print(f" Acurácia no Teste (hold-out): {accuracy_score(y_test, y_pred):.4f}")
print("="*60)

print("\n Relatório de Classificação (Teste):")
print(classification_report(y_test, y_pred,
                            target_names=list(class_mapping.keys()),
                            digits=4))

In [ ]:
print(" Melhores Parâmetros:")
print(random_search.best_params_)

print(f"\n Melhor Acurácia (CV 5 folds): {random_search.best_score_:.4f}")

best_log_model = random_search.best_estimator_

In [ ]:
print(" Melhores Parâmetros:")
print(random_search.best_params_)

print(f"\n Melhor Acurácia (média 5-Fold CV): {random_search.best_score_:.4f}")

best_log_model = random_search.best_estimator_

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded,
    test_size=0.2, random_state=42, stratify=y_encoded
)

y_pred = best_log_model.predict(X_test)

print(f"\n Acurácia no Conjunto de Teste: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório Completo:")
print(classification_report(y_test, y_pred,
                          target_names=class_mapping.keys()))

# Salvando o modelo

In [ ]:
import joblib
import pickle
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = ['Gender', 'family_history', 'FAVC', 'CAEC',
                    'SMOKE', 'SCC', 'CALC', 'MTRANS']
numerical_cols = ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False,
                              drop='first'),
         categorical_cols)
    ],
    remainder='passthrough'
)

preprocessor.fit(X)

print(" Features geradas:", preprocessor.transform(X.head(1)).shape[1])

joblib.dump(preprocessor, "preprocessor.pkl", compress=3)

with open("xgboost_model.pkl", "wb") as f:
    pickle.dump(best_model_xg, f, protocol=4)

with open("class_mapping.pkl", "wb") as f:
    pickle.dump(class_mapping, f, protocol=4)

print(" Salvo com sucesso!")
print(os.listdir())